# knaif skill workbench

Type an utterance, press **Run**, read the plan and the command it renders.

**Not an acceptance instrument.** No corpus, no floors, no safety gate. `just eval-accept`
is the bar and nothing here can move it. A good result here is a reason to run the eval
suite, not a reason to publish.

Open with `uv run jupyter lab notebooks/skill_workbench.ipynb` — the widget JavaScript
ships in the venv and JupyterLab serves it locally. VS Code fetches it from a CDN and will
ask permission to do so.


## 1 · Setup


In [ ]:
# The bench's logic lives in notebooks/shared/workbench/ as ordinary modules, and those
# modules get edited while the bench is open. Without this, a fix lands on disk and the
# kernel keeps running the copy it imported at startup — which reads exactly like the fix
# not working. Costs a re-import per cell run; worth it.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks" / "shared"))

from workbench import inventory, selectors, ui

# The bench's scratch. A real run copies sandbox/fixtures/<skill>/ in here and only
# ever writes to the copies — overwriting a fixture would silently change what every
# later eval run measures. Safe to delete; the next run recreates what it needs.
SCRATCH = ROOT / "sandbox" / "workbench"
SCRATCH.mkdir(parents=True, exist_ok=True)

inv = inventory.scan(ROOT)
print(
    f"{sum(len(inv['models'][g]) for g in ('published','curated','experimental'))} models"
    f" · {len(inv['builds'])} builds · {len(inv['skills'])} skills"
)

## 2 · Pick what to run

Three axes, each scoped to where it is real. **Compute** is a *build* picker because CUDA
versus Vulkan is compiled in, not switched at run time — and each build is labelled by what
it reports, never by its directory name. **force CPU** is the one lever that moves both
runtimes.


In [ ]:
sel = selectors.build_widgets(inv).selection

## 3 · Run

Type, press **Run**, read the plan. The model stays loaded between runs, so another
phrasing costs inference only.

- **verbose** — off shows the plan, the command, where it ran and how long. On adds the
  full plan JSON and the raw stdout/stderr. Toggling it re-renders what already ran; it
  does not re-run inference.
- **runs** — above 1 repeats the utterance and reports n / mean / p50 / p95. Every
  percentile is a figure some run actually took.

**Change any dropdown and just press Run again — no cell needs re-executing.** The
selectors write straight into `sel`, which Run reads fresh each time. Changing the model,
the skill, force CPU or verbose reloads the model once, because each of those is decided
at load; everything else costs nothing.

Re-run **Setup** only after building a new binary or adding a GGUF — that cell is where
the machine is scanned.

**Sample files.** `dry-run` needs none. Switching to **execute for real** copies
`sandbox/fixtures/<skill>/` into the scratch first and runs only on the copies — for
ffmpeg that is `clip.mp4`, `clip_no_audio.mp4`, `clip_4k.mp4`, `audio.mp3` and a few
more. If a skill has no fixtures yet, the run says so and names the command that makes
them: `just eval-fixtures <skill>`.


In [ ]:
bench = ui.console(sel, root=ROOT, sandbox=SCRATCH)

---

### What this machine has

Only needed when a model or build you expected is missing — it names what it looked for.


In [ ]:
print(inventory.summary(inv))